In [2]:
import torch
import pandas as pd

from tqdm.auto import tqdm #progress bar
from transformers import AutoModelForSequenceClassification
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset #smooth loading for dataset
from transformers import AutoTokenizer, AutoConfig

import sqlite3
import torch
from datetime import datetime, UTC
#from scipy.special import softmax
from datasets import Dataset
import re
import numpy as np
import unicodedata

In [3]:
#RETRIEVE DATA
conn1 = sqlite3.connect("starrail_weekly.db")
#post_data = pd.read_sql_query("SELECT * FROM posts",conn1)


In [ ]:
#reddit data
comment_data = pd.read_sql_query("SELECT comment_id, post_id, comment_body FROM comments",conn1)

In [4]:
# youtube data

comment_data = pd.read_sql_query("SELECT comment_id, comment_body, published_at FROM video_comments",conn1)

In [ ]:
# get reddit data not already analyzed
conn = sqlite3.connect("starrail_weekly.db")
query = """SELECT
c.comment_id, 
c.post_id, 
c.comment_body 
FROM comments c
LEFT JOIN roberta_sentiment rs ON c.comment_id = rs.comment_id
WHERE rs.comment_id IS NULL
"""

comment_data = pd.read_sql_query(query,conn)

In [20]:
comment_data

,comment_id,post_id,comment_body
0,t3_1r3ejy9,1r3ejy9,"Oh my God, I didn't think it was possible..."
1,t1_o53r8gi,1r3ejy9,<image>
2,t1_o53u145,1r3ejy9,<image>
3,t1_o53ufm4,1r3ejy9,<image>
4,t1_o53v3ft,1r3ejy9,<image>
...,...,...,...
95965,t1_p7blo6i,1w4z2f5,Obsidian of Repossession. She is the Stonehear...
95966,t1_p7b8s2k,1w4z2f5,Unrelated. Kinda. But… TOPAZ GIRL GET UP ME AN...
95967,t1_p7bay0l,1w4z2f5,There is an actual theme. Like Jades it was es...
95968,t1_p7b9ii8,1w4z2f5,given that they are all based on business and ...


In [ ]:
# First load
#MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"

### Model/Tokenizer save for later
#model.save_pretrained("./roberta_model", from_pt=True) 
#tokenizer.save_pretrained("./roberta_model")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 35870.11it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.00s/it]


In [ ]:
### NaN, <image>, '' counts in comment
print("\nComments NaN:",comment_data["comment_body"].isna().sum(),
      "\nComments with '<image>':",comment_data["comment_body"].str.contains("<image>").sum(),
      "\nComments with '':",comment_data[comment_data["comment_body"]==''].sum())


Comments NaN: 6587 
Comments with '<image>': 28841 
Comments with '': comment_id      t1_o563uttt1_o53upl9t1_o53vbivt1_o5569jdt1_o53...
post_id         1r3ejy91r3ejy91r3ejy91r3ejy91r3ejy91r3ejy91r3e...
comment_body                                                     
dtype: object


In [99]:
len(comment_data)

276354

In [11]:
comment_data.head(10)

,comment_id,post_id,comment_body,text_length
0,t1_oemem9k,1sdul5o,😂,1
1,t1_oaux6yj,1rvr6us,🤓,1
2,t1_p409qqy,1vphtu5,😭,1
3,t1_oqodscf,1u18zya,🤤,1
4,t1_ohtl1y8,1steok1,🙄,1
5,t1_o9mykcf,1rpnqqu,?,1
6,t1_o5e89tn,1r4rogx,🤣,1
7,t1_o6d97gf,1r9jp9o,😭,1
8,t1_onckwew,1tl0u3v,🤣,1
9,t1_odgp3px,1s8f3wf,👌,1


In [6]:
len(comment_data)

30179

In [ ]:
### Cleaning comments
# Removing NaN, <image>, "" (empty string)
# Removing automated comments (post removed for rule breaking, spoilers)
# Comments must have at least 4 words, 3 unique, and average word length > 1.5 (for funny stuff e.g. R E A L, P E A K)

# Filter words for bot comments
AUTOMATED_COMMENTS = [
     r"please keep in mind our spoiler policy",
        r"removed for breaking Rule",
        r"Hey Trailblazer! We see that you posted this with the",
        r"^\[removed]",
        r"^\[deleted]",
        r"Hey Trailblazer! We see that you posted this with the",
        r"Unfortunately, your content had to be removed",
        r"Processing img ",
        r"source",
        r"Shipping Rules",
        r"Thank you for sharing",
        r"Unfortunately, your comment was removed due to",
        r"Unfortunately, your content was removed due to",
        r"Thanks for submitting a Tech Help question",
        r"^\[ Removed by Reddit ]",
        r"^\[Removed by Reddit]",
        r"View link Info",
        r"Other Threads: Weekly Gacha and Drops Megathread",
        r"Honkai: Star Rail's Third Anniversary Grand Opening!",
        r"01001111",
        r"01101111",
        r"View link\nInfo",
        r"\nWeekly Gacha and Drops Megathread",
        r"One day,? after dinner",
        r"Sauce is on my\nInstagram\nand\nTwitter\n!",
        r"FOR REFERENCE:\nPlease be noted that these definitions",
        r"🔄📅👶🦢➕📅⬅️🥱📥👨🦫🪵🌱",
        r"--- -. .",
]

automated_regex = "|".join(AUTOMATED_COMMENTS)

#
def clean_comment_data(df: pd.DataFrame)->pd.DataFrame:
    return (df.dropna(subset=["comment_body","comment_id"])
        .loc[
            lambda df:(df["comment_body"]!="")
            & (df["comment_body"].str.split().str.len()>=4)
            & (df["comment_body"].str.lower().str.split().apply(lambda words: len(set(words)))>=3)
            & (df["comment_body"].str.split().apply(lambda words: sum(len(w) for w in words)/len(words) if words else 0)>1.5)
            & ~df["comment_body"].str.contains("<image>")
            & ~df["comment_id"].str.startswith("t3")
            & ~df["comment_body"].str.contains(automated_regex,case=False,na=False)
            & ~df["comment_body"].str.match(r"^\s*d+[\d\s]*$")
            
        ]
    .assign(text_length=lambda df: df["comment_body"].str.len())
    .sort_values("text_length")
    .reset_index(drop=True)
    )

comment_data=clean_comment_data(comment_data)

In [22]:
### Replace links and mentions (@) with "http", "@"

URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
USER_PATTERN = re.compile(r"@\w+")

def preprocess_comment(text: str, apply_jargon_map: bool = True) -> str:
    """
    Preprocessor for comments.
    Normalizes URLs/@users and maps gacha slang.
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    # Normalize links and mentions
    text = URL_PATTERN.sub("http", text)
    text = USER_PATTERN.sub("@user", text)

    return " ".join(text.split())

# Preprocess
comment_data["comment_body"]=comment_data["comment_body"].apply(preprocess_comment)


In [23]:
def normalize_unicode(text: str) -> str:
    # Converts fancy stylized fonts (e.g. ℌ𝔢𝔯𝔱𝔞) to standard Latin characters
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')

# Apply to your text column before BERTopic fitting
comment_data["comment_body"] = comment_data['comment_body'].apply(normalize_unicode)

In [24]:
comment_data=comment_data.drop_duplicates(subset=["comment_body"])

In [ ]:

# Regex Dictionary to convert game/gacha slang to conventional terms for RoBERTa
GACHA_SLANG_MAP = {
    # --- PULL & LUCK ---
    r"\b(lost\s+the\s+50/50|lost\s+50/50)\b": "unlucky failure",
    r"\b(won\s+the\s+50/50|won\s+50/50)\b": "lucky victory",
    r"\b(hard\s+pity|hit\s+hard\s+pity)\b": "maximum expensive cost",
    r"\b(early\s+5\s*star|early\s+5\*)\b": "lucky early reward",
    r"\b(shafted|got\s+shafted)\b": "scammed",
    r"\b(pity\s+break|spooked|off-banner)\b": "unwanted failure",
    r"\b(building\s+pity)\b": "foolish risk mistake",

    # --- BUZZWORDS ---
    r"\b(cooked|im\s+cooked|account\s+is\s+cooked)\b": "ruined terrible outcome",
    r"\b(mid|super\s+mid)\b": "mediocre underwhelming bad",
    r"\b(peak|absolute\s+peak)\b": "outstanding supreme quality",
    r"\b(copium|hopium)\b": "delusional false hope",
    r"\b(doompost|doomposting|doomposter|doomposters)\b": "extreme toxic complaining",
        
    # --- META & BALANCE ---
    r"\b(powercreep|powercrept|powercreeping|power\s+creep)\b": "unbalanced degradation",
    r"\b(benched|bench\s+unit)\b": "useless abandoned",
    r"\b(nerfed|nerf)\b": "weakened",
    r"\b(buffed|buff)\b": "strengthened",
    r"\b(bricked|bricked\s+account)\b": "ruined state",
    r"\b(min-rolled|min\s+roll|flat\s+stats)\b": "terrible useless stats",
    r"\b(dead\s+week|dead\s+patch)\b": "boring empty content",
    r"\b(wheelchair|wheelchaired)\b":"misleading",
    r"\b(meta)\b":"powerful optimal",

    # --- COMPOUND NOTATIONS & DUPES ---
    r"\b([ce][0-6][sr][0-5])\b": "max upgraded powerful unit", # Matches E6S1, C6R5, etc.
    r"\b(e6|c6|max\s+dupe)\b": "fully upgraded character",
    #r"\b(god\s+roll|godroll)\b": "perfect supreme quality",
    r"\b(must\s+pull|must-pull)\b": "essential top recommendation",
    r"\b(skip\s+banner|easy\s+skip)\b": "unimpressive skippable"
}

def create_processed_text_column(df: pd.DataFrame) -> pd.DataFrame:
    """Generates processed_comment_body from comment_body using regex replacements."""
    # Work on a lowercase copy for replacement matching
    processed = df["comment_body"].astype(str).str.lower()

    for pattern, replacement in GACHA_SLANG_MAP.items():
        processed = processed.str.replace(pattern, replacement, regex=True)

    df["processed_comment_body"] = processed
    return df

comment_data = create_processed_text_column(comment_data)

In [26]:
comment_data

,comment_id,post_id,comment_body,text_length,processed_comment_body
0,t1_p6x5ifc,1w2yjen,Yes it is,11,yes it is
1,t1_olejrvv,1tb6o8c,( ),11,( )
6,t1_p1fl6sm,1ve3kd7,All of them,13,all of them
7,t1_p6syot9,1w2jodl,DM me as well,13,dm me as well
8,t1_oldet7v,1tatwly,I can fix her,13,i can fix her
...,...,...,...,...,...
13014,t1_p6duhu8,1w0cd87,"Look, I won't deny that all of this political ...",3140,"look, i won't deny that all of this political ..."
13015,t1_p7b4i5q,1w4y1b9,"oh boy, okay. so i'm f2p. i think hsr is more ...",3410,"oh boy, okay. so i'm f2p. i think hsr is more ..."
13016,t1_p7a6yhn,1w4rr01,Game Designer here. I'll write down a couple o...,3697,game designer here. i'll write down a couple o...
13017,t1_p727svj,1w3or97,we need to keep E0S0 as the gold standard for ...,3903,we need to keep max upgraded powerful unit as ...


In [27]:
dataset = Dataset.from_pandas(comment_data)

In [15]:
### Load presaved model
model_path = "./roberta_model"

model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
### Pipeline setup
device = 0 if torch.cuda.is_available() else -1

sentiment_pipe = pipeline(
    model=model,
    tokenizer=tokenizer,
    task = "sentiment-analysis",
    top_k=None,  # Returns pos, neutral, neg scores per text
    device=device,
    truncation=True,
    max_length=256,
    dtype=torch.float16 #to speedup model runtime
)

### Batch Inference
batch_size = 64
raw_results = []

for out in tqdm(
    sentiment_pipe(KeyDataset(dataset, 'processed_comment_body'), batch_size=batch_size),
    total=len(comment_data),
    desc="Running Sentiment Analysis"
):
    raw_results.append(out)


Running Sentiment Analysis:   0%|          | 0/12660 [00:00<?, ?it/s]

In [ ]:
### Extract values and format to table for storing
import numpy as np
records = []
analyzed_at = datetime.now(UTC).isoformat()

for idx, (index, row) in enumerate(comment_data.iterrows()):
    comment_results = raw_results[idx]
    
    # Map probability scores
    scores = {item["label"].lower(): item["score"] for item in comment_results}
    
    pos_score = scores.get("positive", 0.0)
    neu_score = scores.get("neutral", 0.0)
    neg_score = scores.get("negative", 0.0)
    
    # Sentiment label (choose highest of 3)
    compound_score = pos_score - neg_score
    sentiment_label = max(scores, key=scores.get)

    # Net sentiment labeling using different conditions
    conservative_conditions = [compound_score>=0.25,compound_score<=-0.25]
    standard_conditions = [compound_score>=0.1,compound_score<=-0.1]
    
    choices = ['positive','negative']

    conservative_label = np.select(conservative_conditions,choices, default="neutral").item()
    standard_label = np.select(standard_conditions,choices, default="neutral").item()
    
    records.append({
        "comment_id": row["comment_id"],    
        "post_id": row["post_id"],
        "comment_body": row["comment_body"],
        "processed_comment_body": row["processed_comment_body"],
        "pos": round(pos_score, 4),
        "neutral": round(neu_score, 4),
        "neg": round(neg_score, 4),
        "compound": round(compound_score, 4),
        "sentiment_label": sentiment_label,
        "conservative_label":conservative_label,
        "standard_label":standard_label,
        "time_analyzed": analyzed_at
    })

sentiment_df = pd.DataFrame(records)

In [30]:
sentiment_df

,comment_id,post_id,comment_body,processed_comment_body,pos,neutral,neg,compound,sentiment_label,conservative_label,standard_label,time_analyzed
0,t1_p6x5ifc,1w2yjen,Yes it is,yes it is,0.5517,0.4345,0.0138,0.5379,positive,positive,positive,2026-09-06T04:14:22.430166+00:00
1,t1_olejrvv,1tb6o8c,( ),( ),0.1186,0.8134,0.0680,0.0506,neutral,neutral,neutral,2026-09-06T04:14:22.430166+00:00
2,t1_p1fl6sm,1ve3kd7,All of them,all of them,0.1250,0.7856,0.0894,0.0355,neutral,neutral,neutral,2026-09-06T04:14:22.430166+00:00
3,t1_p6syot9,1w2jodl,DM me as well,dm me as well,0.1797,0.8003,0.0200,0.1596,neutral,neutral,positive,2026-09-06T04:14:22.430166+00:00
4,t1_oldet7v,1tatwly,I can fix her,i can fix her,0.2741,0.6054,0.1205,0.1536,neutral,neutral,positive,2026-09-06T04:14:22.430166+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...
12655,t1_p6duhu8,1w0cd87,"Look, I won't deny that all of this political ...","look, i won't deny that all of this political ...",0.0116,0.1811,0.8074,-0.7958,negative,negative,negative,2026-09-06T04:14:22.430166+00:00
12656,t1_p7b4i5q,1w4y1b9,"oh boy, okay. so i'm f2p. i think hsr is more ...","oh boy, okay. so i'm f2p. i think hsr is more ...",0.6836,0.2825,0.0339,0.6498,positive,positive,positive,2026-09-06T04:14:22.430166+00:00
12657,t1_p7a6yhn,1w4rr01,Game Designer here. I'll write down a couple o...,game designer here. i'll write down a couple o...,0.1699,0.7948,0.0353,0.1347,neutral,neutral,positive,2026-09-06T04:14:22.430166+00:00
12658,t1_p727svj,1w3or97,we need to keep E0S0 as the gold standard for ...,we need to keep max upgraded powerful unit as ...,0.0471,0.6943,0.2586,-0.2115,neutral,neutral,negative,2026-09-06T04:14:22.430166+00:00


In [31]:
print("Pos:",sum(sentiment_df["sentiment_label"]=="positive"),
"Neg:",sum(sentiment_df["sentiment_label"]=="negative"),
"Neutral:",sum(sentiment_df["sentiment_label"]=="neutral"))

# Pos: 59773 Neg: 75581 Neutral: 141000 # Regex transformed

Pos: 2219 Neg: 5181 Neutral: 5260


In [32]:
print("Pos:",sum(sentiment_df["standard_label"]=="positive"),
"Neg:",sum(sentiment_df["standard_label"]=="negative"),
"Neutral:",sum(sentiment_df["standard_label"]=="neutral"))

# Pos: 100297 Neg: 120645 Neutral: 55412 # Regex transformed

Pos: 3499 Neg: 7228 Neutral: 1933


In [33]:
print("Pos:",sum(sentiment_df["conservative_label"]=="positive"),
"Neg:",sum(sentiment_df["conservative_label"]=="negative"),
"Neutral:",sum(sentiment_df["conservative_label"]=="neutral"))

# Pos: 74441 Neg: 93959 Neutral: 107954 # Regex transformed

Pos: 2650 Neg: 6236 Neutral: 3774


In [35]:
### Store data to database
sentiment_df.to_sql("roberta_sentiment",conn, if_exists="append", index=False)

12660

In [36]:
conn.commit()
conn.close()

In [22]:
### TABLE SCHEMA

cur.execute("""CREATE TABLE roberta_sentiment (
                sentiment_id INTEGER PRIMARY KEY AUTOINCREMENT,
                comment_id TEXT UNIQUE,
                post_id TEXT,
                comment_body TEXT,
                processed_comment_body TEXT,
                pos REAL,
                neutral REAL,
                neg REAL,
                compound REAL,
                sentiment_label TEXT,
                conservative_label TEXT,
                standard_label TEXT,
                time_analyzed DATETIME,
                FOREIGN KEY (comment_id) REFERENCES comments(comment_id) ON UPDATE CASCADE
)""")